# 01 · Memory foundations
### Why agents forget, and what "memory" really means

**Level 100 to 200**

**The story.** You are building an assistant for business banking relationship managers (RMs). A customer, Priya Sharma, runs a freight company and contacts the bank many times a year: by phone, secure message, in a branch. Every RM and every AI agent she meets should already know who she is, what she prefers, and what the bank promised her. She should never have to repeat herself.

This series builds that capability step by step, on AWS Bedrock, with the Strands Agents SDK and the open-source mem0 library.

| # | Notebook | Level | You will learn |
|---|----------|-------|----------------|
| **01** | **Memory foundations** (this one) | 100 to 300 | The three kinds of agent memory. How Strands handles the first two. How mem0 handles the third. Then the foundations of a customer context vault: many sources, entity resolution, a typed memory model, conflicts, time and scopes. |
| 02 | Integration patterns | 200 to 400 | Three ways to wire mem0 into a Strands agent. Then one policy-enforced vault shared by many agents and channels: access control, hybrid retrieval, tools, handoff and the Context Pack. |
| 03 | Evaluating memory | 300 to 400 | A layered evaluation framework: with-memory vs without-memory, LLM-as-judge, latency and cost. Then the vault measured at four levels, including simulated customers. |
| 04 | Production quality | 400 to 500 | Failure modes, quality gates, success criteria, governance. Then operating the vault: lifecycle, erasure, observability and the managed-service mapping. |

**Prerequisites.** AWS credentials with Bedrock access to Claude Haiku 4.5, Claude Sonnet 4.5 and Titan Text Embeddings v2 in `us-east-1` (set `AWS_REGION` to change it), and `pip install -r requirements.txt`. Every cell runs against real models. Nothing is mocked. This notebook takes about 6 minutes and roughly 45 model calls. Parts 1 to 4 are the fundamentals; Part 5 is the vault and can be read on its own once Part 4 makes sense.

**How to read it.** Each part starts with a question, the code answers it, and a short "what just happened" explains the result. Versions used here: `mem0ai 2.0.20`, `strands-agents 1.55.0`. Both libraries changed a lot in 2026, so several things you will read in older blog posts no longer apply. Those differences are called out where they matter.

In [1]:
import json, sys, time, shutil
from importlib.metadata import version
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))   # makes the local `memlab` helper package importable
import memlab                          # sets MEM0_TELEMETRY=false before mem0 is imported
from memlab.config import REGION, AGENT_MODEL_ID, EMBED_MODEL_ID, DATA_DIR, build_model, build_memory, mem0_config

print(f"mem0ai {version('mem0ai')} | strands-agents {version('strands-agents')} | region {REGION}")
print(f"agent + extraction model: {AGENT_MODEL_ID}")
print(f"embedding model:          {EMBED_MODEL_ID}")
print(f"local data dir:           {DATA_DIR}")

mem0ai 2.0.20 | strands-agents 1.55.0 | region us-east-1
agent + extraction model: us.anthropic.claude-haiku-4-5-20251001-v1:0
embedding model:          amazon.titan-embed-text-v2:0
local data dir:           /path/to/mem0-strands-notebooks/.mem0_data


## Part 1 · An agent with no memory

A Strands `Agent` is a model plus a list of messages. That list, `agent.messages`, is the agent's entire mind. Let's watch Priya talk to the assistant, then watch what happens on her next call.

In [2]:
from strands import Agent

SYSTEM_PROMPT = '''You are an assistant for business banking relationship managers.
Answer briefly. If you do not know something about the customer, say so plainly instead of guessing.'''

agent = Agent(model=build_model(), system_prompt=SYSTEM_PROMPT, callback_handler=None)

print(agent("Hi, I'm Priya Sharma from Sharma Logistics in Parramatta. Please only contact me by SMS, "
            "never by phone during the day. I'm usually driving."))
print(agent("Quick check: how should the bank contact me?"))
print(f"\nMessages the agent is carrying: {len(agent.messages)}")

Got it, Priya. I've noted that you prefer SMS contact only—no phone calls during the day since you're usually driving. I'll make sure that's recorded in your communication preferences.

Is there anything I can help you with today regarding your business banking?

Based on what you just told me: **SMS only, never by phone during the day** since you're usually driving.

Is that correct?


Messages the agent is carrying: 4


Within one conversation the agent remembers, because every message is still in `agent.messages` and is sent to the model on every turn.

Now Priya calls back the next day. In production that is a new request, a new process, a new `Agent` object.

In [3]:
next_call = Agent(model=build_model(), system_prompt=SYSTEM_PROMPT, callback_handler=None)
print(next_call("It's Priya Sharma again. How should the bank contact me?"))

I don't have information about Priya Sharma's contact details in my current context. To help you properly, I would need to look up your account information or you could tell me your preferred contact method (phone, email, etc.).

How would you like the bank to reach you?



The agent has no idea. Nothing was wrong with the model; the information simply lived in a Python list that is gone.

### Three kinds of memory

It helps to give the three things people call "memory" different names, because they are solved by different tools.

| Kind | What it is | Lifetime | Question it answers | Tool in this series |
|------|-----------|----------|---------------------|---------------------|
| **Working memory** | The messages of the current run | One invocation | "What did she say two turns ago?" | `agent.messages` + a conversation manager |
| **Session memory** | The same conversation, resumable later | One conversation, across restarts | "Where were we when the call dropped?" | Strands `SessionManager` |
| **Long-term memory** | Facts distilled from *all* conversations | The customer relationship | "What does Priya prefer? What did we promise her in June?" | mem0 |

In the customer context vault vocabulary: working and session memory are the *transcript*, long-term memory is where the *episodic*, *semantic*, *preference* and *commitment* layers live. Parts 2 and 3 cover the first two quickly. Part 4 is the heart of this notebook.

## Part 2 · Working memory: the conversation window

Every turn re-sends the whole history. Two consequences: cost grows with every turn, and eventually the context overflows. Let's measure the growth with real token counts.

In [4]:
agent = Agent(model=build_model(), system_prompt=SYSTEM_PROMPT, callback_handler=None)

turns = [
    "I'm Priya Sharma, Sharma Logistics, Parramatta. Contact me by SMS only, never phone during the day.",
    "We run 14 trucks and have an equipment loan with you.",
    "Our EFTPOS terminal at the depot keeps dropping out.",
    "We are thinking about opening a Newcastle depot in November.",
    "Please note my accountant is Deepak Rao.",
    "What contact preference did I give you at the start?",
]

rows, previous_total = [], 0
for i, text in enumerate(turns, 1):
    result = agent(text)
    total = result.metrics.accumulated_usage["inputTokens"]   # accumulates over the agent's lifetime
    rows.append((i, len(agent.messages), total - previous_total))
    previous_total = total

print(f"{'turn':>4} {'messages in history':>20} {'input tokens this turn':>24}")
for r in rows:
    print(f"{r[0]:>4} {r[1]:>20} {r[2]:>24}")
print("\nLast answer:", str(result).strip())

turn  messages in history   input tokens this turn
   1                    2                       70
   2                    4                      165
   3                    6                      250
   4                    8                      367
   5                   10                      463
   6                   12                      515

Last answer: SMS only, never phone during the day.


Input tokens climb every turn even though each new message is one sentence: the model is re-reading the whole conversation. In a long support session that is the dominant cost.

Strands manages this with a **conversation manager**. The default `SlidingWindowConversationManager` keeps the last `window_size` messages (40 by default). Watch what a tight window does to recall.

In [5]:
from strands.agent.conversation_manager import SlidingWindowConversationManager

windowed = Agent(model=build_model(), system_prompt=SYSTEM_PROMPT, callback_handler=None,
                 conversation_manager=SlidingWindowConversationManager(window_size=4))

for text in turns[:-1]:
    windowed(text)
print(f"Messages kept in history: {len(windowed.messages)} (window_size=4)")
print(windowed(turns[-1]))

Messages kept in history: 4 (window_size=4)
I don't have any record of a contact preference you gave me at the start of our conversation. This is the first time we've spoken.

Could you remind me what your preferred contact method is?



The window kept the conversation cheap and the contact preference fell out of it. `SummarizingConversationManager` is the other option: when the context overflows, it replaces the oldest messages with a model-written summary. That keeps *some* of the old content, in compressed and lossy form.

**Takeaway.** Working memory is recency-bound by design. It is the right tool for the current task and the wrong tool for anything that must survive the conversation.

## Part 3 · Session memory: pick up where you left off

A **session manager** persists `agent.messages` (and `agent.state`) as they are created, so a new process can rebuild the same agent. Here we use `FileSessionManager` on local disk. `S3SessionManager` is the same idea with an S3 bucket, and is the usual production choice when the runtime is stateless.

In [6]:
from strands.session.file_session_manager import FileSessionManager

SESSIONS_DIR = DATA_DIR / "sessions"
session_id = "priya-call-2026-09-08"
shutil.rmtree(SESSIONS_DIR / f"session_{session_id}", ignore_errors=True)   # clean start for this demo

agent = Agent(model=build_model(), system_prompt=SYSTEM_PROMPT, callback_handler=None,
              agent_id="rm_assistant",
              session_manager=FileSessionManager(session_id=session_id, storage_dir=str(SESSIONS_DIR)))
agent("I'm Priya Sharma from Sharma Logistics. SMS only please, never phone during the day.")
agent("Also, our depot EFTPOS terminal keeps dropping out.")

# The call drops. Later, a different process picks it up: a brand-new Agent, same session_id and agent_id.
resumed = Agent(model=build_model(), system_prompt=SYSTEM_PROMPT, callback_handler=None,
                agent_id="rm_assistant",
                session_manager=FileSessionManager(session_id=session_id, storage_dir=str(SESSIONS_DIR)))
print(f"Messages restored from disk: {len(resumed.messages)}")
print(resumed("Sorry, the line dropped. What two things did I tell you already?"))

Messages restored from disk: 4
No problem. You told me two things:

1. **Contact preference:** SMS only, no daytime calls
2. **Issue:** Your depot EFTPOS terminal keeps dropping out



In [7]:
for path in sorted((SESSIONS_DIR / f"session_{session_id}").rglob("*.json")):
    print(path.relative_to(SESSIONS_DIR))

one_message = next((SESSIONS_DIR / f"session_{session_id}").rglob("message_*.json"))
print("\nOne stored message:\n", json.dumps(json.loads(one_message.read_text()), indent=1)[:600])

session_priya-call-2026-09-08/agents/agent_rm_assistant/agent.json
session_priya-call-2026-09-08/agents/agent_rm_assistant/messages/message_0.json
session_priya-call-2026-09-08/agents/agent_rm_assistant/messages/message_1.json
session_priya-call-2026-09-08/agents/agent_rm_assistant/messages/message_2.json
session_priya-call-2026-09-08/agents/agent_rm_assistant/messages/message_3.json
session_priya-call-2026-09-08/agents/agent_rm_assistant/messages/message_4.json
session_priya-call-2026-09-08/agents/agent_rm_assistant/messages/message_5.json
session_priya-call-2026-09-08/session.json

One stored message:
 {
 "message": {
  "role": "user",
  "content": [
   {
    "text": "Also, our depot EFTPOS terminal keeps dropping out."
   }
  ],
  "tracking_id": "231f4612-a755-4878-aed1-4206b77a937f"
 },
 "message_id": 2,
 "redact_message": null,
 "created_at": "2026-09-29T18:44:17.584268+00:00",
 "updated_at": "2026-09-29T18:44:17.584271+00:00"
}


Session memory solves "resume this conversation". It does **not** solve the original problem, for three reasons:

1. It is a verbatim transcript keyed by session. Priya's June call and her September call are different sessions and know nothing about each other.
2. To use it you load the whole transcript. Nothing is distilled, so the cost problem from Part 2 comes back.
3. It cannot answer "what does Priya prefer?" across channels without reading everything she has ever said.

For that we need a different kind of store: one that holds *facts*, not *messages*.

## Part 4 · Long-term memory with mem0

mem0 does one job well: it turns conversations into a small set of self-contained facts, stores each fact with an embedding, and retrieves the few that matter for a query. Three moving parts, all of which you choose:

```
   conversation ──► LLM (extract facts) ──► embedder ──► vector store   (+ SQLite history for audit)
   query        ──► embedder ──► vector store (nearest facts) ──► ranked facts
```

Here everything runs on AWS: Claude on Bedrock extracts, Titan Text Embeddings v2 embeds, and a local FAISS index stores vectors (swapped for OpenSearch in production, notebook 04). Here is the full configuration.

In [8]:
print(json.dumps(mem0_config("nb01"), indent=2))

{
  "llm": {
    "provider": "aws_bedrock",
    "config": {
      "model": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
      "temperature": 0.1,
      "max_tokens": 2000,
      "aws_region": "us-east-1"
    }
  },
  "embedder": {
    "provider": "aws_bedrock",
    "config": {
      "model": "amazon.titan-embed-text-v2:0",
      "embedding_dims": 1024,
      "aws_region": "us-east-1"
    }
  },
  "vector_store": {
    "provider": "faiss",
    "config": {
      "collection_name": "nb01",
      "path": "/path/to/mem0-strands-notebooks/.mem0_data/faiss/nb01",
      "embedding_model_dims": 1024,
      "distance_strategy": "cosine"
    }
  },
  "history_db_path": "/path/to/mem0-strands-notebooks/.mem0_data/history/nb01.db",
  "custom_instructions": null
}


What each part means:

- **`llm`**: the model that reads a conversation and writes facts. Provider string is `aws_bedrock`. Cross-region inference profile ids (`us.anthropic...`) work. `aws_region` must be set explicitly, otherwise mem0 silently falls back to `us-west-2`.
- **`embedder`**: Titan v2 with `embedding_dims: 1024`. The vector store must be told the same number, under a *different* key, `embedding_model_dims`. Mismatch is the most common setup error.
- **`vector_store`**: FAISS with `distance_strategy: cosine`, so every score is a cosine similarity in `[0, 1]`. The default (`euclidean`) produces scores that are hard to reason about.
- **`history_db_path`**: a SQLite file where mem0 records every add, update and delete. This is your audit trail.
- **`custom_instructions`**: extra rules for the extractor (used in notebook 04). Older tutorials mention `custom_fact_extraction_prompt` and `custom_update_memory_prompt`; those keys no longer exist in mem0 2.x.
- Not visible here: `MEM0_TELEMETRY=false`, set in `memlab/__init__.py`. mem0 sends anonymous usage telemetry by default; a bank turns it off.

In [9]:
memory = build_memory("nb01", fresh=True)   # fresh=True wipes any earlier run of this notebook
print(type(memory).__name__, "ready")

Model us.anthropic.claude-haiku-4-5-20251001-v1:0 may not be available in region us-east-1
The 'faiss' vector store does not support keyword search. Hybrid (BM25) scoring will be disabled and search will use semantic similarity only. To enable hybrid search, switch to a store with keyword_search support (e.g. qdrant, elasticsearch, pgvector).


Memory ready


Two warnings appear above and both are safe to ignore. mem0 checks the model id against `ListFoundationModels`, which does not list cross-region inference profiles (`us.anthropic...`), so it warns even though the model works. And FAISS has no keyword index, so mem0 tells you it will rank by semantic similarity only; OpenSearch and pgvector add BM25 keyword scoring on top.

### 4.1 · `add()`: from transcript to facts

Here is the transcript of Priya's first call. The customer is the `user`, the banker is the `assistant`. We store it under `user_id="cust_priya"`: the memory belongs to the customer, not to the conversation.

In [10]:
first_call = [
    {"role": "user", "content": "Hi, I'm Priya Sharma, I run Sharma Logistics, a freight business in Parramatta with 14 trucks. "
                                "We have our business transaction account and an equipment loan with you."},
    {"role": "assistant", "content": "Thanks Priya. How can I help today?"},
    {"role": "user", "content": "Our EFTPOS terminal at the depot keeps dropping out. Also, please only contact me by SMS, "
                                "never by phone during the day, I'm usually driving."},
    {"role": "assistant", "content": "Understood. I've logged a terminal fault and I'll have a replacement terminal shipped to the depot "
                                     "by Thursday. I'll confirm by SMS."},
]

started = time.perf_counter()
added = memory.add(first_call, user_id="cust_priya", metadata={"channel": "phone", "session_id": "s1"})
print(f"add() took {time.perf_counter() - started:.1f}s and produced {len(added['results'])} memories:\n")
for item in added["results"]:
    print(f"  [{item['event']}] {item['memory']}")

add() took 9.7s and produced 5 memories:

  [ADD] User's name is Priya Sharma and she runs Sharma Logistics, a freight business in Parramatta with 14 trucks
  [ADD] Sharma Logistics has a business transaction account and an equipment loan with the bank
  [ADD] Priya prefers to be contacted by SMS only, never by phone during the day because she is usually driving
  [ADD] Sharma Logistics' EFTPOS terminal at the depot was experiencing intermittent dropouts as of September 29, 2026
  [ADD] A replacement EFTPOS terminal was arranged to be shipped to Sharma Logistics' depot by Thursday, October 3, 2026, with SMS confirmation to follow


**What just happened.** One call to `add()` ran this pipeline:

1. mem0 fetched the last 10 messages already stored for this scope (to resolve "it", "the terminal") and the 10 existing memories most similar to the new text (to avoid duplicates).
2. **One** LLM call, with a long extraction prompt, returned a JSON list of facts. Each fact is written to be self-contained ("Priya's EFTPOS terminal at the depot..." rather than "the terminal..."), and relative dates are resolved to absolute ones.
3. Each fact was embedded, checked against existing memories by hash, and inserted into FAISS with your metadata attached.
4. A row per fact went into the SQLite history table.

Notice the dates in the facts: "by Thursday" became an absolute date. mem0 open source resolves relative dates against *today*, and has no parameter for telling it when a conversation actually happened. When you ingest historical transcripts, put the date inside the text (notebook 03 does exactly that).

A note on versions: the mem0 research paper and most tutorials describe a two-call design where a second LLM call decides to ADD, UPDATE or DELETE existing memories. mem0 2.x open source replaced that with the single "additive" call you just saw, and `add()` only ever reports `ADD` events. Part 4.5 shows why that matters.

Let's look at the instructions the extractor actually follows.

In [11]:
from mem0.configs.prompts import ADDITIVE_EXTRACTION_PROMPT
print(ADDITIVE_EXTRACTION_PROMPT.strip()[:1400], "\n...")

# ROLE

You are a Memory Extractor — a precise, evidence-bound processor responsible for extracting rich, contextual memories from conversations. Your sole operation is ADD: identify every piece of memorable information and produce self-contained, contextually rich factual statements.

You extract from BOTH user and assistant messages. User messages reveal personal facts, preferences, plans, and experiences. Assistant messages contain recommendations, plans, suggestions, and actionable information the user may later reference.

Accuracy and completeness are critical. Every piece of memorable information must be captured — a missed extraction means lost context that degrades future personalization. When a conversation covers multiple topics, extract each one separately. Do not let a dominant topic cause you to miss secondary information.

# INPUTS

## New Messages

The current conversation turn(s) with "role" (user/assistant) and "content".

Both roles contain extractable information:
-

Two design choices to notice: facts are extracted from **both** user and assistant messages (so the bank's promise about the Thursday terminal became a memory), and the prompt insists on *contextually rich* facts over atomic fragments. Both matter for a banking use case, where the promise is often the most important fact in the call.

What if the same transcript is ingested twice, say by a retry?

In [12]:
again = memory.add(first_call, user_id="cust_priya", metadata={"channel": "phone", "session_id": "s1"})
print(f"Second add() of the same transcript produced {len(again['results'])} new memories.")

Second add() of the same transcript produced 0 new memories.


Zero. The extractor saw the existing memories and skipped them, and the hash check would have caught exact duplicates anyway. Ingestion is idempotent for identical input, which makes retries safe.

### 4.2 · `search()`: the few facts that matter

Retrieval is an embedding lookup, so it is fast and has no LLM in the loop. `filters` is **mandatory** and must name at least one of `user_id`, `agent_id`, `run_id`. `explain=True` shows how the score was built.

In [13]:
def show(results):
    for item in results["results"]:
        score = f"{item['score']:.3f}" if item.get("score") is not None else "     "
        print(f"  {score}  {item['memory']}")

started = time.perf_counter()
hits = memory.search("how does the customer want to be contacted", filters={"user_id": "cust_priya"}, top_k=3, explain=True)
print(f"search() took {(time.perf_counter() - started) * 1000:.0f} ms\n")
show(hits)
print("\nscore breakdown of the top hit:", json.dumps(hits["results"][0]["score_details"], indent=1))

search() took 305 ms

  0.171  Priya prefers to be contacted by SMS only, never by phone during the day because she is usually driving

score breakdown of the top hit: {
 "semantic_score": 0.17086312174797058,
 "bm25_score": 0.0,
 "entity_boost": 0.0,
 "raw_score": 0.17086312174797058,
 "max_possible_score": 1.0,
 "final_score": 0.17086312174797058,
 "threshold": 0.1
}


In [14]:
print("Query: 'what did the bank promise her?'")
show(memory.search("what did the bank promise her?", filters={"user_id": "cust_priya"}, top_k=3))
print("\nQuery: 'what does her business do?'")
show(memory.search("what does her business do?", filters={"user_id": "cust_priya"}, top_k=3))

Query: 'what did the bank promise her?'
  0.183  Sharma Logistics has a business transaction account and an equipment loan with the bank
  0.113  A replacement EFTPOS terminal was arranged to be shipped to Sharma Logistics' depot by Thursday, October 3, 2026, with SMS confirmation to follow

Query: 'what does her business do?'
  0.228  User's name is Priya Sharma and she runs Sharma Logistics, a freight business in Parramatta with 14 trucks
  0.116  Sharma Logistics has a business transaction account and an equipment loan with the bank


**Reading the scores.** On FAISS the score is the cosine similarity between the query and the fact, blended with an *entity boost* when a named entity in the query (a person, a company, a place) also appears in the memory. Vector stores that support keyword search (OpenSearch, pgvector, Qdrant) add a BM25 term as well, and the score is normalised back into `[0, 1]`. Two knobs shape the result list:

- `top_k` (default 20): how many facts to return.
- `threshold` (default 0.1): the minimum semantic score. Lower it to see weaker matches; raise it to cut noise.

Scores around 0.2 to 0.5 are normal for Titan embeddings and are not a sign of a bad match. Absolute values depend on the embedding model; what matters is the ranking. And the ranking is not always right: look at the query about what the bank *promised*. The fact about the replacement terminal is the real answer, yet the account fact ranked above it, because embeddings match wording ("with the bank") more than intent ("a promise"). Retrieval quality is measurable, and notebook 03 measures it.

One API detail that breaks a lot of older code (including, at the time of writing, the `mem0_memory` tool shipped in `strands-agents-tools`): passing `user_id` directly to `search()` or `get_all()` is rejected.

In [15]:
try:
    memory.search("contact preference", user_id="cust_priya")
except ValueError as error:
    print("ValueError:", error)

ValueError: Top-level entity parameters frozenset({'user_id'}) are not supported in search(). Use filters={'user_id': '...'} instead.


### 4.3 · Scopes: whose memory is it?

Every memory carries up to three identities, and `filters` selects on them. Choosing the right scope is the single most important modelling decision.

| Scope | Use it for | Example |
|-------|-----------|---------|
| `user_id` | Facts about the **customer**, across every conversation and channel | "Priya prefers SMS" |
| `agent_id` | Facts the **agent** learned about doing its job, across customers | "Terminal faults are resolved by shipping a replacement and confirming by SMS" |
| `run_id` | Facts scoped to **one conversation**, disposable | "Currently discussing the July statement" |

`metadata` is yours: tag memories with channel, session, purpose or source system, and filter on them later. mem0's filter language has operators (`eq`, `ne`, `in`, `gte`, `contains`, `AND`, `OR`, ...), but **each vector store implements its own subset**. The local FAISS store implements only two forms: an exact value, and a plain list meaning "any of these". An operator it does not implement is not an error: it silently matches nothing. The cell below shows the difference.

In [16]:
print("All memories for Priya that came from a phone call:")
show(memory.get_all(filters={"user_id": "cust_priya", "channel": "phone"}, top_k=100))

print("\nA plain list means 'any of these' on FAISS:")
show(memory.get_all(filters={"user_id": "cust_priya", "channel": ["phone", "branch"]}, top_k=100))

print("\nThe operator form of the same filter, which FAISS does not implement:")
print(f"  {len(memory.get_all(filters={'user_id': 'cust_priya', 'channel': {'in': ['phone', 'branch']}}, top_k=100)['results'])} results")

All memories for Priya that came from a phone call:
         User's name is Priya Sharma and she runs Sharma Logistics, a freight business in Parramatta with 14 trucks
         Sharma Logistics has a business transaction account and an equipment loan with the bank
         Priya prefers to be contacted by SMS only, never by phone during the day because she is usually driving
         Sharma Logistics' EFTPOS terminal at the depot was experiencing intermittent dropouts as of September 29, 2026
         A replacement EFTPOS terminal was arranged to be shipped to Sharma Logistics' depot by Thursday, October 3, 2026, with SMS confirmation to follow

A plain list means 'any of these' on FAISS:
         User's name is Priya Sharma and she runs Sharma Logistics, a freight business in Parramatta with 14 trucks
         Sharma Logistics has a business transaction account and an equipment loan with the bank
         Priya prefers to be contacted by SMS only, never by phone during the day because

The operator form returned nothing, and nothing warned you. On OpenSearch or pgvector the same filter works. Two consequences for a bank: a filter that silently matches nothing looks exactly like "the customer has no such memories", and a policy enforced by an unsupported filter enforces nothing. Test every filter you rely on against the store you run, and do range filters such as dates in application code on FAISS (Part 5 does exactly that).

### 4.4 · `update()`, `delete()`, `history()`: memory is data you own

Every memory has an id. You can read it, correct it, remove it, and see its full change history. In a regulated setting the history table is what lets you answer "what did the system believe about this customer on this date?"

In [17]:
preference = next(m for m in memory.get_all(filters={"user_id": "cust_priya"}, top_k=100)["results"] if "SMS" in m["memory"])
print("Before:", preference["memory"])

memory.update(preference["id"], text=preference["memory"] + " (confirmed by the customer in writing)")
print("After: ", memory.get(preference["id"])["memory"])

print("\nHistory of this memory:")
for event in memory.history(preference["id"]):
    print(f"  {event['created_at'][:19]}  {event['event']:<7} {event['new_memory'][:90]}")

Before: Priya prefers to be contacted by SMS only, never by phone during the day because she is usually driving
After:  Priya prefers to be contacted by SMS only, never by phone during the day because she is usually driving (confirmed by the customer in writing)

History of this memory:
  2026-09-29T18:44:27  ADD     Priya prefers to be contacted by SMS only, never by phone during the day because she is us
  2026-09-29T18:44:27  UPDATE  Priya prefers to be contacted by SMS only, never by phone during the day because she is us


### 4.5 · When the world changes: the contradiction test

Two months later Priya changes her mind. Watch closely what the store does.

In [18]:
memory.add([{"role": "user", "content": "Actually, update my contact preference. Phone calls are fine now, any time after 4pm. "
                                        "SMS is no longer needed."}],
           user_id="cust_priya", metadata={"channel": "secure_message", "session_id": "s4"})

print("Everything the store now believes about contacting Priya:")
show(memory.search("contact preference", filters={"user_id": "cust_priya"}, top_k=5))

Everything the store now believes about contacting Priya:
  0.551  Priya's contact preference has changed: phone calls are now acceptable after 4pm. SMS is no longer required as her preferred contact method.
  0.212  Priya prefers to be contacted by SMS only, never by phone during the day because she is usually driving (confirmed by the customer in writing)


Both facts are there. The new one usually ranks higher, because the extractor wrote it as a *transition* ("updated her contact preference..."), which matches "contact preference" queries well. But the old fact was not removed or marked superseded, and a query phrased differently could surface it first.

This is the most important thing to understand about memory quality: **the library gives you storage and retrieval. Keeping the memory *true* is your design responsibility.** An RM who phones Priya at 11am because the assistant surfaced the stale fact is exactly the failure this project exists to prevent. Notebook 03 measures how often it happens; notebook 04 shows three ways to fix it.

### 4.6 · Two more tools in the box

**Verbatim storage.** `infer=False` skips the LLM and stores each message as-is. Use it for things that must not be paraphrased (a customer's exact complaint wording, a disclosure that was read out) or when facts were already extracted upstream.

In [19]:
verbatim = memory.add([{"role": "user", "content": "I want it on record that I was told the annual fee would be waived for the first year."}],
                      user_id="cust_priya", infer=False, metadata={"kind": "verbatim", "session_id": "s2"})
print(verbatim["results"][0])

{'id': 'b5892abc-d15c-4e13-8dc6-fccccd0c88f5', 'memory': 'I want it on record that I was told the annual fee would be waived for the first year.', 'event': 'ADD', 'actor_id': None, 'role': 'user'}


**Procedural memory.** With `agent_id` and `memory_type="procedural_memory"`, mem0 summarises a conversation into a *procedure*: what the agent did and how, stored under the agent rather than the customer. It is how the assistant gets better at its job over time, and the raw material for the vault's *procedural* layer.

In [20]:
handled = memory.add(
    first_call,
    agent_id="rm_assistant",
    memory_type="procedural_memory",
)
print(handled["results"][0]["memory"][:900])

## Summary of the agent's execution history

**Task Objective**: Create a procedural memory record of the customer interaction with Priya Sharma from Sharma Logistics.

**Progress Status**: 100% complete — Customer information and service requests fully documented.

---

1. **Agent Action**: Received and processed customer introduction and business context.
   
   **Action Result**:
   ```
   Customer Name: Priya Sharma
   Business Name: Sharma Logistics
   Location: Parramatta
   Business Type: Freight business
   Fleet Size: 14 trucks
   Existing Accounts: Business transaction account and equipment loan
   ```
   
   **Key Findings**: 
   - Customer is the owner/operator of Sharma Logistics
   - Business has established banking relationship with multiple account types
   - Customer actively manages operations (frequently driving)
   
   **Current Context**: Customer profile established


## Part 5 · From a memory to a customer context vault

**Level 200 to 300.** Part 4 gave us a memory: flat facts about one customer from one kind of source. A bank needs more than that. The use-case analysis names the hard parts, and this part builds the foundations for the first four, on the same customer:

| Problem space | The question | Where in this series |
|---|---|---|
| 1 · Interaction data foundation | How do calls, notes, cases, chats, emails and system events become one interaction model, with lineage? | **5.1** here; resilience and upstream deletions in 04 |
| 2 · Customer and entity resolution | How do we know every item belongs to the right customer, business and relationship? | **5.2** here |
| 3 · Fact conflict resolution | How do facts coexist when they disagree, without silently overwriting history? | **5.4** here |
| 4 · Memory model and lifecycle | What is remembered, as which type, what is derived, what expires, what is never stored? | **5.3** and **5.6** here; lifecycle in 04 |
| 5 · Retrieval and relevance | How do we find the few memories that matter now? | 02 |
| 6 · Context Pack construction | What is the bounded contract between memory and agent behaviour? | 02 |
| 7 · Grounding, trust, explainability | How does a banker know where a memory came from and whether it is true? | 5.3 here (the fields); 02 (the pack) |
| 8 · Evaluation and business value | Is it better, safe enough and worth it? | 03 and 04 |
| 9 · Memory layer technology | Managed, composable or hybrid; which scopes; which stores? | 5.6 here (scopes); 04 (architecture) |

The scenario grows too. Priya's five conversations are joined by what a bank actually holds about her: system-of-record profiles (KYC, CRM, the product directory), a relationship manager's note, a complaint case, a Virtual RM chat, a merchant platform event and two emails. There is also an email from an accountant who works for *both* of our customers, and a note whose customer id matches nobody. "Today", for the scenario, is 27 September 2026. Everything is in `data/vault_dataset.json`, and every function below is the same code as `memlab/vault.py`, which notebooks 02 to 04 import.

In [21]:
import hashlib, re
from dataclasses import dataclass, field
from typing import Literal
from pydantic import BaseModel, Field
from memlab.vault import load_vault_dataset
from memlab.evaluation import load_dataset

dataset, vault_dataset = load_dataset(), load_vault_dataset()
TODAY = vault_dataset["today"]
directory = vault_dataset["party_directory"]
raw = [dict(s, kind="session", customer_id=cid) for cid, c in dataset["customers"].items() for s in c["sessions"]] + vault_dataset["sources"]
print(f"{len(raw)} raw records from {len({r['kind'] for r in raw})} kinds of source, plus "
      f"{sum(len(f) for f in vault_dataset['system_of_record'].values())} system-of-record facts. Today is {TODAY}.")
for kind in sorted({r["kind"] for r in raw}):
    print(f"  {kind:<16} {sum(r['kind'] == kind for r in raw)}")

16 raw records from 7 kinds of source, plus 15 system-of-record facts. Today is 2026-09-27.
  call_transcript  1
  banker_note     3
  complaint        1
  email_summary    2
  merchant_event   1
  session          7
  vrm_chat         1


### 5.1 · Many sources, one interaction model (problem space 1)

Each source system has its own shape: a transcript has turns, a RM note has an author and free text, a complaint has an owner and a service commitment, a merchant event has a timestamp and a type. The **memory adaptor** normalises all of them into one `Interaction` with the same fields, using one rule per source kind. Three fields matter more than they look:

- `source_ref` is the **lineage key** back to the original record (`banker_notes:CN-40112`, `complaints:C-9912`). Every memory derived from the interaction carries it, which is what makes a memory explainable, traceable and, when the source is corrected or deleted, removable (notebook 04).
- `occurred_on` is when it *happened*, not when we stored it. Part 4 showed that mem0 resolves "last Friday" against today; the adaptor puts the real date in front of the extractor.
- `audience` and `purposes` are set **by the rule for the source**, not by a model: a relationship manager's note is `rm_only`, a complaint is `internal`, a conversation with the customer is shareable with the customer. `authoritative` marks system events and systems of record, whose facts are *verified*.

In [22]:
@dataclass
class Interaction:
    """One normalised unit of customer history, whatever system it came from."""
    source_system: str            # contact_centre, vrm, banker_notes, complaints, merchant_ops, email, kyc, crm, pds
    native_id: str                # the id in the originating system
    occurred_on: str              # event date, YYYY-MM-DD: when it happened, not when we stored it
    channel: str                  # phone, branch, web_chat, vrm, rm_note, case, system, email, profile
    kind: str                     # transcript | note | case | event | email
    text: str                     # rendered for the extractor, speakers labelled
    identifiers: dict             # what the source carried: cif, party_id, merchant_id, email, customer_id
    author: str | None = None
    audience: str = "customer"    # who may see memories from it: customer | internal | rm_only
    purposes: tuple = ("servicing", "relationship_management", "complaints")
    authoritative: bool = False   # system events and systems of record produce verified memories

    @property
    def source_ref(self) -> str:
        """The lineage key: every memory derived from this interaction carries it, so it can be traced and deleted."""
        return f"{self.source_system}:{self.native_id}"


SPEAKER = {"user": "Customer", "assistant": "Banker", "customer": "Customer", "vrm": "Virtual RM"}


def _lines(turns, who, what) -> str:
    return "\n".join(f"{SPEAKER[t[who]]}: {t[what]}" for t in turns)


# One rule per source kind: this is the "rules engine in the adaptor" from the reference design.
ADAPTER_RULES = {
    "session": lambda s: Interaction(
        "contact_centre", s["session_id"], s["date"], s["channel"], "transcript",
        _lines(s["messages"], "role", "content"), {"customer_id": s["customer_id"]}),
    "call_transcript": lambda s: Interaction(
        "contact_centre", s["call_id"], s["date"], s["channel"], "transcript",
        _lines(s["messages"], "role", "content"), {"party_id": s["party_id"]}, author=s.get("banker")),
    "vrm_chat": lambda s: Interaction(
        "vrm", s["conversation_id"], s["date"], "vrm", "transcript",
        _lines(s["turns"], "speaker", "text"), {"party_id": s["party_id"]}, author="Virtual RM (AI agent)"),
    "banker_note": lambda s: Interaction(
        "banker_notes", s["note_id"], s["created"], "rm_note", "note",
        f"Note by {s['author']}: {s['text']}", {"cif": s["cif"]}, author=s["author"],
        audience="rm_only", purposes=("servicing", "relationship_management")),
    "complaint": lambda s: Interaction(
        "complaints", s["case_id"], s["lodged"], "case", "case",
        f"Complaint {s['case_id']} lodged {s['lodged']} via {s['channel']} ({s['category']}), owner {s['owner']}, "
        f"status {s['status']}. {s['summary']} Service commitment: {s['sla']}",
        {"cif": s["cif"]}, author=s["owner"], audience="internal", purposes=("complaints", "servicing")),
    "merchant_event": lambda s: Interaction(
        "merchant_ops", s["event_id"], s["timestamp"][:10], "system", "event",
        f"Merchant system event {s['event_id']} ({s['type']}) on {s['timestamp'][:10]}: {s['detail']}",
        {"merchant_id": s["merchant_id"]}, author="merchant platform", authoritative=True),
    "email_summary": lambda s: Interaction(
        "email", s["message_id"], s["date"], "email", "email",
        f"Email from {s['from']} to {s['to']}, subject '{s['subject']}': {s['summary']}",
        {"email": s["from"]}, author=s["from"]),
}


def normalise(raw: dict) -> Interaction:
    """Turn a raw record from any source into an Interaction. Unknown kinds fail loudly rather than being guessed."""
    if raw["kind"] not in ADAPTER_RULES:
        raise ValueError(f"no adapter rule for source kind {raw['kind']!r}")
    return ADAPTER_RULES[raw["kind"]](raw)

In [23]:
interactions = sorted((normalise(r) for r in raw), key=lambda i: i.occurred_on)
print(f"{'occurred':<11} {'source_ref':<28} {'channel':<14} {'audience':<9} {'verified':<9} text")
for i in interactions:
    print(f"{i.occurred_on:<11} {i.source_ref:<28} {i.channel:<14} {i.audience:<9} {str(i.authoritative):<9} {i.text[:60].replace(chr(10), ' ')}...")

occurred    source_ref                   channel        audience  verified  text
2026-06-10  contact_centre:s1            phone          customer  False     Customer: Hi, this is Priya Sharma calling from Sharma Logis...
2026-06-11  banker_notes:CN-40112             rm_note        rm_only   False     Note by Alex Nguyen (relationship manager): Follow-up after ...
2026-06-24  contact_centre:m1            email          customer  False     Customer: Hi, Marcus Chen from Chen's Bakery in Marrickville...
2026-07-02  contact_centre:s2            secure_message customer  False     Customer: Hi, I need to dispute a charge on the business car...
2026-07-28  contact_centre:s3            branch         customer  False     Customer: Morning. Priya Sharma, Sharma Logistics. I was nea...
2026-08-05  contact_centre:m2            phone          customer  False     Customer: Morning, Marcus Chen, Chen's Bakery. I know I said...
2026-08-14  contact_centre:s4            phone          customer  False   

### 5.2 · Whose interaction is this? (problem space 2)

Before anything is extracted, every interaction must be attached to exactly one customer. Sources identify customers in different ways (a CIF number, a party id, a merchant id, an email address), and business banking adds a harder case: people and entities connected to *several* customers. Deepak Rao is the external accountant for both Sharma Logistics and Chen's Bakery, so an email from him says nothing, by its sender alone, about whose memory it belongs in.

The rule is simple and strict: **resolve by identifier; for a third party, only an unambiguous business name in the content decides; otherwise quarantine.** A wrong attachment is not a retrieval error, it is a privacy breach, and it is invisible afterwards because the memory looks exactly like any other.

In [24]:
def _short_business(name: str) -> str:
    return re.sub(r"\s+pty\.?\s+ltd\.?$", "", name.strip(), flags=re.I)


def resolve_customer(interaction: Interaction, directory: dict) -> tuple[str | None, str]:
    """Return (customer_id, reason). Never guess: an unknown or ambiguous identifier goes to quarantine (None)."""
    ids = interaction.identifiers
    if ids.get("customer_id"):
        return ids["customer_id"], "exported with a customer id"
    for cid, c in directory["customers"].items():
        for key, known in (("cif", [c["cif"]]), ("party_id", c["party_ids"]), ("merchant_id", c["merchant_ids"]), ("email", c["emails"])):
            if ids.get(key) and ids[key] in known:
                return cid, f"{key} {ids[key]} belongs to {c['name']}"
    # A known third party (an accountant, a broker) can be linked to several customers. The sender alone does not say
    # which customer the interaction is about; only an unambiguous business name in the content does.
    for party in directory["associated_parties"].values():
        if ids.get("email") and ids["email"] in party["emails"]:
            named = [cid for cid in party["linked_customers"]
                     if _short_business(directory["customers"][cid]["business"]).lower() in interaction.text.lower()]
            if len(named) == 1:
                return named[0], (f"third party {party['name']} is linked to {len(party['linked_customers'])} customers; "
                                  f"the content names only {directory['customers'][named[0]]['business']}")
            return None, f"third party {party['name']} is linked to several customers and the content does not say which"
    return None, "no identifier matched a known customer"

In [25]:
resolved = []
for i in interactions:
    customer_id, reason = resolve_customer(i, directory)
    resolved.append((i, customer_id))
    print(f"{i.source_ref:<28} -> {customer_id or 'QUARANTINE':<12} {reason}")

contact_centre:s1            -> cust_priya   exported with a customer id
banker_notes:CN-40112             -> cust_priya   cif CIF-100231 belongs to Priya Sharma
contact_centre:m1            -> cust_marcus  exported with a customer id
contact_centre:s2            -> cust_priya   exported with a customer id
contact_centre:s3            -> cust_priya   exported with a customer id
contact_centre:m2            -> cust_marcus  exported with a customer id
contact_centre:s4            -> cust_priya   exported with a customer id
contact_centre:s5            -> cust_priya   exported with a customer id
contact_centre:CALL-88213    -> cust_priya   party_id PTY-7781 belongs to Priya Sharma
banker_notes:CN-40977             -> cust_marcus  cif CIF-100874 belongs to Marcus Chen
complaints:C-9912            -> cust_priya   cif CIF-100231 belongs to Priya Sharma
vrm:VRM-310554               -> cust_priya   party_id PTY-7781 belongs to Priya Sharma
merchant_ops:ME-2209184      -> cust_priya   merchant_

Two lines to read closely. `email:EM-77120` came from the accountant who serves both customers; the content names Chen's Bakery, so it went to Marcus and *not* to Priya, even though Priya's memory also mentions Deepak Rao. And `banker_notes:CN-41230` matched nobody, so it waits in quarantine for a person, rather than being attached to the nearest-looking customer.

### 5.3 · A memory model: typed, sourced, verified (problem spaces 4 and 7)

The SEA scope document proposes five memory types. Here is how each is represented, and which fields every memory carries so that a banker can tell a source fact from an interpretation:

| Layer | What it holds | How it is produced | Scope |
|---|---|---|---|
| **semantic** | Stable facts: entities, people and roles, products, fleet | Extracted, or written directly from a system of record | customer (`user_id`) |
| **preference** | How the customer wants to be treated | Extracted; a change is a new value for the same slot | customer |
| **episodic** | What happened: faults, disputes, credits, corrections | Extracted, with its date | customer |
| **commitment** | What the *bank* promised, by when, and whether it is done | Extracted, with a due date and an open/closed state | customer |
| **procedural** | How the team resolves things | Written by the team (playbooks), or mem0's procedural memory (Part 4.6) | agent (`agent_id`) |
| **temporal** | How the relationship changes over time ("three terminals in three months") | *Derived* from episodic records, recomputed (notebook 04) | customer |

On every record: `asserted_by` (customer, bank, agent, system, third party), `kind` (**direct** statement or **derived** interpretation), `verification` (verified by a system, or unverified), `confidence`, `source_ref`, `observed_on`, `audience`, `purposes` and `policy_version`. That is the "every material memory should carry..." list from problem space 7, as data.

mem0's own extractor returns plain sentences, so a typed schema needs our own extraction step: one structured-output call per interaction, whose results are written to mem0 with `infer=False` and the fields as metadata. (In AgentCore Memory terms this is a *self-managed strategy*: built-in strategies have a fixed output schema.) Two design choices make it work. A controlled vocabulary of **attribute slots** (`fleet_size`, `contact_preference`, ...) turns "is this an update?" into a comparison instead of a judgement. And the extractor sees the **vault's current state**, open items and current slot values, so it can say which open commitment an interaction fulfils.

In [26]:
LAYERS = ("semantic", "preference", "episodic", "commitment", "procedural", "temporal")

# A controlled vocabulary of attribute "slots". A new value for a slot is what makes an older memory outdated, so
# naming the slot at extraction time turns knowledge updates into a data operation instead of a model judgement.
ATTRIBUTES = {
    "business_name": "the business's trading or legal name",
    "directors": "who the directors or owners are",
    "related_entity": "related companies, trusts or premises owners",
    "products_held": "the bank products the business holds",
    "fleet_size": "number of vehicles or trucks",
    "contact_preference": "how and when the customer wants to be contacted",
    "accountant": "the customer's external accountant and what to copy them on",
    "depot_location": "where the business operates from",
    "expansion_plan": "a planned new site, with its timing",
    "terminal_status": "whether the merchant terminal currently works",
    "overdraft_limit": "the current overdraft limit",
    "relationship_manager": "the bank's relationship manager for the customer",
}


class Relation(BaseModel):
    subject: str = Field(description="An entity, e.g. 'Deepak Rao'")
    predicate: str = Field(description="snake_case relationship, e.g. 'accountant_of', 'owns', 'director_of'")
    object: str = Field(description="An entity, e.g. 'Sharma Logistics'")


class MemoryRecord(BaseModel):
    """One typed memory. The text is what gets embedded; everything else becomes metadata."""
    text: str = Field(description="One self-contained sentence with absolute dates and the customer's or business's name")
    layer: Literal["semantic", "preference", "episodic", "commitment"] = Field(
        description="semantic: a stable fact; preference: how they want to be treated; episodic: something that happened; "
                    "commitment: a promise the BANK made, with a due date")
    attribute: str | None = Field(default=None, description="The attribute slot this record sets, from the vocabulary given, or null")
    value: str | None = Field(default=None, description="The slot's value, short and canonical, e.g. '16 trucks'. Reuse the exact existing value if unchanged")
    entities: list[str] = Field(default_factory=list, description="Named people, businesses, places, products and reference numbers")
    relations: list[Relation] = Field(default_factory=list, description="Relationships between entities stated in the interaction")
    due_date: str | None = Field(default=None, description="For commitments: the due date, YYYY-MM-DD")
    status: Literal["open", "closed", "n/a"] = Field(default="n/a", description="open for a commitment or an unresolved issue or request; closed if it records completion; n/a otherwise")
    closes: list[str] = Field(default_factory=list, description="Ids of OPEN ITEMS from the vault state that this record fulfils, resolves or cancels")
    asserted_by: Literal["customer", "bank", "agent", "system", "third_party"] = Field(description="Who stated it")
    kind: Literal["direct", "derived"] = Field(description="direct: stated in the source; derived: an interpretation, e.g. that someone seems frustrated")
    corrects: bool = Field(default=False, description="True if the customer is correcting something the bank or an AI agent got wrong")
    confidence: float = Field(description="0 to 1: how sure you are the record is a faithful reading of the source")


class Extraction(BaseModel):
    records: list[MemoryRecord]

In [27]:
EXTRACTION_POLICY = """You turn one interaction between a bank and a business customer into typed memory records for the bank's
customer context vault. Bankers and AI agents will rely on these records, so be exact and be conservative.

Write each record as ONE self-contained sentence with absolute dates, resolving words like "last Friday" against the
interaction date. Name the customer or business in each sentence.

Keep:
- semantic: stable facts about the customer and the business (entities, people and their roles, products, fleet, sites).
- preference: how the customer wants to be contacted or treated. A changed preference is a new value for the same attribute.
- episodic: what happened: faults, disputes, complaints, payments, credits, corrections, and the outcome of earlier items.
- commitment: every promise the BANK made to the customer ("I'll send", "we will", "it will be credited by"), with what,
  by when (due_date) and any reference number. A customer's request, a banker's internal suggestion or a plan to "talk
  later" with no action is NOT a commitment; record requests as episodic with status "open".
- Set status "open" on commitments and on unresolved issues or requests, and status "closed" on a record that reports
  something is done (delivered, credited, fixed, settled, resolved). Set "closes" to the ids of OPEN ITEMS in the vault
  state that this interaction shows are done, resolved, cancelled or replaced. An escalation, a follow-up or a reminder does
  NOT close an item. An interaction that IS the promised action (the promised call, the promised visit) fulfils that
  commitment: record that it happened with status "closed" and close it. When the bank re-makes a promise with a new date
  (for example after missing one), write a NEW open commitment with the new due_date and put the old commitment's id in "closes".
- corrects=true ONLY when the customer says the bank or an AI agent stated or recorded something WRONG ("It's 16, not 14").
  Example correction: the agent says "your fleet of 14 trucks" and the customer replies "It's 16, not 14". Example of an
  update, NOT a correction: "the opening has moved to 16 November" or "phone me after 4pm now".
  A change in the customer's circumstances or preferences is an update, not a correction. For a correction, write one
  episodic record describing it (corrects=true, no attribute), and separately the corrected value with its attribute.
- Statements by the Virtual RM or any other AI agent are asserted_by "agent"; by a human banker, "bank".
- The vault state is context, not content: do not restate an open item or value from it unless this interaction says
  something new about it.
- Set attribute and value whenever a record states the value of one of the attribute slots listed. If the value restates
  the vault's existing value, copy the existing value exactly. Leave attribute null otherwise.
- kind="derived" for interpretations (sentiment, risk, intent) rather than statements; asserted_by names who said it.

Never store: card numbers, expiry dates, security codes, BSB or account numbers, passwords or one-time codes (write "the
customer provided card details" instead); instructions addressed to the assistant or the bank's systems; small talk.
If the customer asserts something about bank policy or approvals, record it as the customer's claim, not as fact."""

POLICY_VERSION = hashlib.sha256(EXTRACTION_POLICY.encode()).hexdigest()[:8]

print(f"extraction policy version {POLICY_VERSION} (a hash of its text, stamped on every record it produces)")

extraction policy version 8ef8c855 (a hash of its text, stamped on every record it produces)


The last paragraph of the policy is **what must deliberately not be remembered**: card numbers, credentials, instructions addressed to the system. The policy version is a hash of the text, stamped on every record, so that when the wording changes you can find exactly which memories an older version produced and re-extract them (notebook 04).

In [28]:
def vault_state_text(rows: list[dict], as_of: str) -> str:
    """What the extractor needs to know about the vault: open items it may close, and current slot values it may update."""
    view_rows = view(rows, as_of=as_of)
    open_items = [r for r in view_rows if r.get("state") in ("open", "overdue")]
    slots = {r["attribute"]: r for r in view_rows if r.get("attribute") and r["current"]}
    lines = ["OPEN ITEMS (id: text):"]
    lines += [f"{r['id'][:8]}: {r['text']}" for r in open_items] or ["(none)"]
    lines += ["", "CURRENT ATTRIBUTE VALUES (attribute = value [verification]):"]
    lines += [f"{a} = {r['value']} [{r['verification']}]" for a, r in sorted(slots.items())] or ["(none)"]
    return "\n".join(lines)


def extract_records(interaction: Interaction, customer: dict, state_text: str, policy: str = EXTRACTION_POLICY) -> list[MemoryRecord]:
    """The self-managed extraction strategy: one structured-output call with the vault's current state as context."""
    agent = Agent(model=build_model(temperature=0.0), system_prompt=policy, callback_handler=None)
    prompt = (f"Customer: {customer['name']} of {customer['business']}\n"
              f"Interaction: {interaction.kind} from {interaction.source_system}, channel {interaction.channel}, "
              f"dated {interaction.occurred_on}, author {interaction.author or 'n/a'}\n\n"
              f"Attribute vocabulary: {json.dumps(ATTRIBUTES)}\n\nVault state before this interaction:\n{state_text}\n\n"
              f"Interaction content:\n{interaction.text}")
    return agent(prompt, structured_output_model=Extraction).structured_output.records

### 5.4 · Writing without overwriting (problem space 3)

The MVP rule for fact conflicts is explicit: *do not silently overwrite; store conflicting facts with timestamps and provenance; let them coexist where the platform cannot safely resolve the conflict; present them with context; do not write back to source systems.* `write_record` implements exactly that, with one rule per case:

| The new record... | ...against an existing record for the same slot | Result |
|---|---|---|
| has the same value | any | No duplicate: the existing record is **reconfirmed** and the new source is added to `confirmed_by` |
| has a new value | unverified | The old record is **superseded**, with `superseded_by` and `superseded_on`. It stays in the store as history |
| is unverified, with a different value | **verified** (a system of record) | **Both are kept**, linked by `conflicts_with`, and the discrepancy is logged for review. The record is not changed |

Two guards are enforced in code rather than trusted to the prompt, because a model follows instructions most of the time, not every time: only the bank can make a commitment (a customer's "I need it by Friday" becomes an open *request*), and an open item is closed only by a record that reports completion or by a bank re-promise with a new date. Without the second guard, in testing, an email that merely *mentioned* the Newcastle application closed the commitment to send it.

In [29]:
def _norm(value) -> str:
    return re.sub(r"[^a-z0-9]", "", str(value or "").lower())


def _meta(record: MemoryRecord, interaction: Interaction, policy_version: str = POLICY_VERSION) -> dict:
    return {
        "layer": record.layer, "attribute": record.attribute, "value": record.value,
        "entities": record.entities, "relations": [r.model_dump() for r in record.relations],
        "due_date": record.due_date, "status": record.status, "asserted_by": record.asserted_by, "kind": record.kind,
        "corrects": record.corrects, "confidence": record.confidence,
        "verification": "verified" if interaction.authoritative else "unverified",
        "source_system": interaction.source_system, "source_ref": interaction.source_ref,
        "observed_on": interaction.occurred_on, "channel": interaction.channel, "author": interaction.author,
        "audience": interaction.audience, "purposes": list(interaction.purposes), "policy_version": policy_version,
        "confirmed_by": [],
    }


def write_record(memory, customer_id: str, text: str, meta: dict, rows: list[dict], log: dict) -> str:
    """Write one record, applying the conflict rules against the rows already in the vault. Returns the new id (or the
    id of the existing record it reconfirmed). Rules: same slot and same value -> reconfirm, no duplicate; new value
    over an unverified value -> supersede with a dated link; unverified value against a verified one -> keep both and
    queue the discrepancy for review. Nothing is ever silently overwritten."""
    slot = meta.get("attribute")
    current = [r for r in rows if slot and r.get("attribute") == slot and not r.get("superseded_by")]
    # the same sentence again (an extractor restating an item it saw in the vault state) is a reconfirmation, not a new record
    same_text = [r for r in rows if _norm(r["text"]) == _norm(text) and not r.get("superseded_by")]
    for old in same_text + current:
        if old in same_text or _norm(old.get("value")) == _norm(meta.get("value")):
            confirmed = list(old.get("confirmed_by") or []) + [meta["source_ref"]]
            memory.update(old["id"], metadata={"confirmed_by": confirmed, "last_confirmed_on": meta["observed_on"]})
            old["confirmed_by"] = confirmed
            log["reconfirmed"].append({"id": old["id"], "value": old["value"], "by": meta["source_ref"]})
            return old["id"]
    new_id = memory.add([{"role": "user", "content": text}], user_id=customer_id, infer=False, metadata=meta)["results"][0]["id"]
    rows.append({"id": new_id, "text": text, **meta})
    for old in current:
        if old["verification"] == "verified" and meta["verification"] == "unverified":
            memory.update(new_id, metadata={"conflicts_with": old["id"]})
            rows[-1]["conflicts_with"] = old["id"]
            log["conflicts"].append({"attribute": slot, "record_of": old["value"], "record_source": old["source_ref"],
                                     "stated": meta["value"], "stated_source": meta["source_ref"], "stated_on": meta["observed_on"]})
        else:
            memory.update(old["id"], metadata={"superseded_by": new_id, "superseded_on": meta["observed_on"]})
            old.update(superseded_by=new_id, superseded_on=meta["observed_on"])
            log["superseded"].append({"attribute": slot, "old": old["value"], "new": meta["value"], "on": meta["observed_on"]})
    return new_id


def apply_guards(record: MemoryRecord, interaction: Interaction) -> MemoryRecord:
    """Deterministic rules the model is told but cannot be trusted to follow every time."""
    if interaction.source_system == "vrm" and record.asserted_by == "bank":
        # no human banker speaks in a Virtual RM chat: what "the bank" said there, an AI agent said
        record = record.model_copy(update={"asserted_by": "agent"})
    if record.layer == "commitment" and record.asserted_by not in ("bank", "agent"):
        # only the bank (or its agent) can make a commitment; a customer's "I need X by Y" is an open request
        record = record.model_copy(update={"layer": "episodic", "status": "open"})
    if record.kind == "derived" and record.status == "open":
        # an interpretation ("seems frustrated") is context, not an item anyone can complete
        record = record.model_copy(update={"status": "n/a"})
    return record


def may_close(record: MemoryRecord, old: dict) -> bool:
    """An open item is closed only by a record that reports completion, or by a bank re-promise of a bank commitment.
    A reminder, a request or an escalation that mentions the item does not close it, and an AI agent's reassurance
    ("I'll make sure the team updates this") never replaces a commitment a banker made."""
    reports_completion = record.status == "closed" and record.asserted_by != "agent"
    re_promise = record.layer == "commitment" and record.asserted_by == "bank" and old.get("layer") == "commitment"
    return reports_completion or re_promise


def ingest(memory, interaction: Interaction, customer_id: str, customer: dict, policy: str = EXTRACTION_POLICY) -> dict:
    """Extract and write one interaction. Idempotent: an interaction whose source_ref is already in the vault is skipped.
    `policy` defaults to the current extraction policy; its hash is stamped on every record (re-extraction, notebook 04)."""
    version = POLICY_VERSION if policy == EXTRACTION_POLICY else hashlib.sha256(policy.encode()).hexdigest()[:8]
    rows = vault_rows(memory, customer_id)
    log = {"source_ref": interaction.source_ref, "written": [], "reconfirmed": [], "superseded": [], "conflicts": [], "closed": []}
    if any(r["source_ref"] == interaction.source_ref or interaction.source_ref in (r.get("confirmed_by") or []) for r in rows):
        log["skipped"] = "already ingested"
        return log
    records = extract_records(interaction, customer, vault_state_text(rows, interaction.occurred_on), policy)
    by_short = {r["id"][:8]: r for r in rows}
    for record in records:
        record = apply_guards(record, interaction)
        new_id = write_record(memory, customer_id, record.text, _meta(record, interaction, version), rows, log)
        log["written"].append({"id": new_id, "layer": record.layer, "text": record.text})
        for short in record.closes:
            old = by_short.get(short[:8])
            if old and old.get("status") == "open" and old["id"] != new_id and may_close(record, old):
                memory.update(old["id"], metadata={"status": "closed", "closed_on": interaction.occurred_on, "closed_by": new_id})
                old.update(status="closed", closed_on=interaction.occurred_on, closed_by=new_id)
                log["closed"].append({"id": old["id"], "text": old["text"], "on": interaction.occurred_on})
    return log


def ingest_system_of_record(memory, customer_id: str, facts: list[dict]) -> list[str]:
    """Verified facts from KYC, CRM and product systems: no model call, written in date order through the same write
    rules as everything else, so a later record (a director resigning) supersedes the earlier one with a dated link.
    Relations the system of record declares (director_of, owns) are kept; they become the graph's edges (notebook 02)."""
    rows, log, ids = vault_rows(memory, customer_id), {"reconfirmed": [], "superseded": [], "conflicts": []}, []
    for f in sorted(facts, key=lambda f: f["recorded_on"]):
        meta = {"layer": "semantic", "attribute": f["attribute"], "value": f["value"], "entities": f.get("entities", []),
                "relations": f.get("relations", []), "due_date": None, "status": "n/a", "asserted_by": "system", "kind": "direct",
                "corrects": False, "confidence": 1.0, "verification": "verified", "source_system": f["source_system"],
                "source_ref": f["source_ref"], "observed_on": f["recorded_on"], "channel": "profile", "author": f["source_system"],
                "audience": "customer", "purposes": ["servicing", "relationship_management", "complaints", "credit_decision"],
                "policy_version": "sor", "confirmed_by": []}
        ids.append(write_record(memory, customer_id, f["text"], meta, rows, log))
    return ids

### 5.5 · Views: the vault now, and as of any date

Nothing in the store is ever edited into its "current" form. The current state, and the state on any past date, are **views** computed from the metadata: a record is current unless it was superseded on or before the date, and an open item is open, overdue or closed *as of* the date. This is what answers "what did we know on 1 August?" and what lets a commitment be overdue today without anyone having to mark it.

In [30]:
def vault_rows(memory, scope_id: str, scope: str = "user_id") -> list[dict]:
    """Every record in one scope, flattened: id, text, created_at and all metadata."""
    items = memory.get_all(filters={scope: scope_id}, top_k=1000)["results"]
    return [{"id": m["id"], "text": m["memory"], "created_at": m.get("created_at"), **(m.get("metadata") or {})} for m in items]


def view(rows: list[dict], as_of: str | None = None, today: str = "2026-09-27") -> list[dict]:
    """The vault as it stood on a date (event time). Each row gains `current` and, for open items, a `state`:
    open, overdue or closed, all judged as of that date. Rows observed after the date are left out."""
    date = as_of or today
    out = []
    for r in rows:
        if (r.get("observed_on") or "") > date:
            continue
        row = dict(r)
        superseded_on = r.get("superseded_on")
        row["current"] = not (superseded_on and superseded_on <= date)
        if r.get("status") in ("open", "closed"):
            closed = r.get("status") == "closed" and (r.get("closed_on") or "") <= date
            overdue = not closed and r.get("due_date") and r["due_date"] < date
            row["state"] = "closed" if closed else ("overdue" if overdue else "open")
        out.append(row)
    return sorted(out, key=lambda r: r.get("observed_on") or "", reverse=True)


def conflicts(rows: list[dict]) -> list[str]:
    """Present conflicting facts with their context rather than choosing one (the MVP rule for fact conflicts)."""
    by_id = {r["id"]: r for r in rows}
    notes = []
    for r in rows:
        old = by_id.get(r.get("conflicts_with"))
        if old and r.get("current", True):
            notes.append(f"The {old['source_system']} record ({old['observed_on']}) gives {old['attribute']} as '{old['value']}'. "
                         f"In a later interaction ({r['observed_on']}, {r['source_system']}) the customer stated '{r['value']}'. "
                         f"The stated value is unverified; the record has not been changed.")
    return notes


def render(row: dict) -> str:
    """One line per memory, carrying everything a banker or a model needs to weigh it."""
    tags = [row.get("layer", "?"), f"observed {row.get('observed_on', '?')}", row.get("verification", "?")]
    if row.get("asserted_by") == "customer":
        tags.append("customer-stated")
    if row.get("kind") == "derived":
        tags.append("interpretation")
    if row.get("state"):
        tags.append(row["state"].upper() + (f" due {row['due_date']}" if row.get("due_date") and row["state"] != "closed" else ""))
    if row.get("current") is False:
        tags.append(f"HISTORY, replaced on {row['superseded_on']}")
    if row.get("conflicts_with"):
        tags.append("differs from system of record")
    tags.append(f"source {row.get('source_ref', '?')}")
    return f"[{' | '.join(tags)} | id {row['id'][:8]}] {row['text']}"

Now build the vault. Systems of record first, then the first conversation on its own, so you can compare typed records with the flat facts mem0 extracted from the same call in Part 4.1.

In [31]:
vault = build_memory("nb01_vault", fresh=True)
for customer_id, facts in vault_dataset["system_of_record"].items():
    ingest_system_of_record(vault, customer_id, facts)

first = next(i for i, cid in resolved if i.source_ref == "contact_centre:s1")
started = time.perf_counter()
log = ingest(vault, first, "cust_priya", directory["customers"]["cust_priya"])
print(f"ingest() of {first.source_ref} took {time.perf_counter() - started:.1f}s and wrote {len(log['written'])} records:\n")
rows = {r["id"]: r for r in vault_rows(vault, "cust_priya")}
print(f"{'layer':<11} {'attribute':<19} {'asserted':<9} {'kind':<8} {'status':<7} {'due':<11} text")
for w in log["written"]:
    r = rows[w["id"]]
    print(f"{r['layer']:<11} {str(r['attribute']):<19} {r['asserted_by']:<9} {r['kind']:<8} {r['status']:<7} {str(r['due_date']):<11} {r['text'][:80]}")

Model us.anthropic.claude-haiku-4-5-20251001-v1:0 may not be available in region us-east-1
The 'faiss' vector store does not support keyword search. Hybrid (BM25) scoring will be disabled and search will use semantic similarity only. To enable hybrid search, switch to a store with keyword_search support (e.g. qdrant, elasticsearch, pgvector).


ingest() of contact_centre:s1 took 8.2s and wrote 8 records:

layer       attribute           asserted  kind     status  due         text
semantic    depot_location      customer  direct   n/a     None        Sharma Logistics Pty Ltd operates from a depot in Parramatta, western Sydney.
semantic    None                customer  direct   n/a     None        Priya Sharma of Sharma Logistics Pty Ltd drives one of the trucks herself most d
preference  contact_preference  customer  direct   n/a     None        Priya Sharma of Sharma Logistics Pty Ltd prefers to be contacted by SMS only, no
episodic    None                customer  direct   open    None        The merchant EFTPOS terminal at Sharma Logistics Pty Ltd's Parramatta depot has 
episodic    None                bank      direct   n/a     None        The bank logged a fault with the merchant team for Sharma Logistics Pty Ltd's EF
commitment  terminal_status     bank      direct   open    2026-06-19  The bank committed to ship a repla

Compare with Part 4.1. The same call produced sentences there; here each fact also says what *kind* of memory it is, which slot it sets, who said it, and, for the bank's promise, when it is due and that it is open. Everything downstream (views, conflicts, policy, the Context Pack, evaluation) is built on these fields.

Now every other interaction, in date order. Watch the right-hand columns: closures, supersessions and conflicts are the vault reasoning about time.

In [32]:
build_log = [log]
started = time.perf_counter()
for i, customer_id in resolved:
    if customer_id is None or i.source_ref == first.source_ref:
        continue
    entry = ingest(vault, i, customer_id, directory["customers"][customer_id])
    build_log.append(entry)
    print(f"{i.occurred_on}  {i.source_ref:<28} -> {customer_id:<12} wrote {len(entry['written']):>2}  closed {len(entry['closed'])}  "
          f"superseded {len(entry['superseded'])}  conflicts {len(entry['conflicts'])}  reconfirmed {len(entry['reconfirmed'])}")
print(f"\n{len(build_log)} interactions ingested in {time.perf_counter() - started:.0f}s; 1 quarantined")

2026-06-11  banker_notes:CN-40112             -> cust_priya   wrote  3  closed 0  superseded 0  conflicts 0  reconfirmed 0
2026-06-24  contact_centre:m1            -> cust_marcus  wrote  7  closed 0  superseded 0  conflicts 0  reconfirmed 0
2026-07-02  contact_centre:s2            -> cust_priya   wrote  8  closed 3  superseded 1  conflicts 0  reconfirmed 0
2026-07-28  contact_centre:s3            -> cust_priya   wrote 10  closed 1  superseded 0  conflicts 1  reconfirmed 1
2026-08-05  contact_centre:m2            -> cust_marcus  wrote  6  closed 3  superseded 1  conflicts 0  reconfirmed 0
2026-08-14  contact_centre:s4            -> cust_priya   wrote  9  closed 5  superseded 1  conflicts 0  reconfirmed 3
2026-09-03  contact_centre:s5            -> cust_priya   wrote  7  closed 3  superseded 0  conflicts 0  reconfirmed 2
2026-09-04  contact_centre:CALL-88213    -> cust_priya   wrote  9  closed 2  superseded 1  conflicts 0  reconfirmed 0
2026-09-08  banker_notes:CN-40977             -> cu

In [33]:
print("Supersessions (a new value for the same slot):")
for entry in build_log:
    for s in entry["superseded"]:
        print(f"  {s['on']}  {s['attribute']:<19} '{s['old'][:45]}'  ->  '{s['new'][:45]}'")

print("\nConflicts with a system of record, presented with context rather than resolved:")
for note in conflicts(view(vault_rows(vault, "cust_priya"), today=TODAY)):
    print("  " + note)

print("\nIdempotency: ingesting the complaint again ->",
      ingest(vault, next(i for i, _ in resolved if i.source_ref == "complaints:C-9912"), "cust_priya", directory["customers"]["cust_priya"]).get("skipped"))

Supersessions (a new value for the same slot):
  2026-07-02  terminal_status     'replacement terminal in transit'  ->  'replacement terminal operational'
  2026-08-05  overdraft_limit     '$20,000'  ->  '$35,000'
  2026-08-14  contact_preference  'SMS only, no phone calls during the day'  ->  'phone calls after 4pm'
  2026-09-04  expansion_plan      'Newcastle depot, November 2026'  ->  'Newcastle depot, 2 November 2026'
  2026-09-22  expansion_plan      'Newcastle depot, 2 November 2026'  ->  'Newcastle depot, 16 November 2026'

Conflicts with a system of record, presented with context rather than resolved:
  The crm record (2025-10-06) gives fleet_size as '14 trucks'. In a later interaction (2026-07-28, contact_centre) the customer stated '16 trucks'. The stated value is unverified; the record has not been changed.

Idempotency: ingesting the complaint again -> already ingested


The fleet size is the conflict problem space 3 describes. The CRM record says 14 trucks (the 2025 annual review); Priya said 16 in July and corrected an AI agent to 16 in September. The vault does not decide which is true. It keeps both, links them, and presents the difference, so a banker sees "the record says 14, the customer says 16" rather than whichever sentence happened to rank first. Whether the record gets corrected is a decision for a person and the system of record's owners (notebook 04's review queue), not for an extractor.

Now the views. First, open items today; then the same questions as of two past dates.

In [34]:
rows = vault_rows(vault, "cust_priya")
print(f"OPEN AND OVERDUE ITEMS AS OF {TODAY}:")
for r in view(rows, today=TODAY):
    if r.get("state") in ("open", "overdue"):
        print("  " + render(r))

for as_of, slots in (("2026-08-01", ("fleet_size", "contact_preference")), ("2026-07-15", ())):
    print(f"\nAS OF {as_of}:")
    for r in view(rows, as_of=as_of):
        if (r.get("attribute") in slots and r["current"]) or (not slots and r.get("layer") == "commitment" and r.get("state") in ("open", "overdue")):
            print("  " + render(r))

OPEN AND OVERDUE ITEMS AS OF 2026-09-27:
  [semantic | observed 2026-09-22 | unverified | customer-stated | OPEN | source email:EM-77164 | id 366c2e66] Priya Sharma of Sharma Logistics Pty Ltd requires the merchant facility and terminal at the Newcastle depot operational by Monday 16 November 2026.
  [episodic | observed 2026-09-22 | unverified | customer-stated | OPEN | source email:EM-77164 | id 9b10b23f] Priya Sharma of Sharma Logistics Pty Ltd requested on 2026-09-22 that the merchant facility application be sent to both her and her accountant Deepak Rao.
  [commitment | observed 2026-09-15 | unverified | OVERDUE due 2026-09-25 | source vrm:VRM-310554 | id 5293be05] The Virtual RM noted on 2026-09-15 that Deepak Rao will be copied on the Newcastle merchant paperwork for Priya Sharma of Sharma Logistics Pty Ltd.
  [episodic | observed 2026-09-14 | unverified | customer-stated | OPEN | source complaints:C-9912 | id 36484f13] Priya Sharma of Sharma Logistics Pty Ltd requested a writte

**Reading the views.** As of today the Newcastle merchant application, promised by 25 September, is *overdue*, and nobody had to flag it. As of 15 July the fee waiver was already overdue (due 9 July), which is what Priya would have experienced. And as of 1 August the contact preference is still SMS only, because the change came on 14 August.

Extraction is a model call, so the exact records differ a little from run to run, and not every closure is caught: you may see a request that a later interaction replaced still marked open (for example the 2 November and the 16 November "needed by" dates side by side). The vault shows such items dated and sourced rather than hiding them, which is the safe failure, and notebook 03 measures how often it happens.

One subtlety worth naming: these are *event-time* views, "what the bank had been told by that date". In this notebook every record was stored today, so *record time* (when the system learned it) is the same for all of them. In production the two differ, and mem0's `history()` table is the record-time log that answers "what did the system show the banker on that date?". A bank needs both.

### 5.6 · Whose memory is it? Customer, banker, agent, conversation (problem space 9)

Part 4.3 introduced scopes. The vault uses four, and the priority from the pilot scope is customer memory first, then banker memory:

| Scope | Holds | Who may read it | Example |
|---|---|---|---|
| Customer (`user_id=cust_...`) | The typed records above | Channels acting for that customer, subject to audience and purpose (notebook 02) | "Deepak Rao is copied on loan paperwork" |
| Banker (`user_id=banker_...`) | The banker's own working memory: how they like briefs, what they are preparing | Only that banker | "I'm preparing Priya's annual review for 6 October" |
| Agent (`agent_id=...`) | Procedural playbooks shared across customers | Bank-side agents | "When a terminal fails a second time, book a technician" |
| Conversation (`run_id=...`) | Scratch notes for one conversation, with an expiry date | That conversation | "Currently comparing the two Newcastle quotes" |

A banker's note can mention a customer. The vault indexes those mentions, because a customer's erasure request must reach them too (notebook 04 shows what happens when it does not).

In [35]:
def write_banker_note(memory, banker_id: str, note: dict, directory: dict) -> str:
    """The banker's own working memory: scoped to the banker, private to them, and indexed by the customers it mentions
    so that a customer's erasure request can find it (notebook 04)."""
    mentions = [cid for cid, c in directory["customers"].items()
                if c["name"].lower() in note["text"].lower() or _short_business(c["business"]).lower() in note["text"].lower()]
    meta = {"layer": "preference" if not mentions else "episodic", "audience": "banker_private", "owner": banker_id,
            "mentions_customers": mentions, "observed_on": note["date"], "source_system": "banker_note",
            "source_ref": f"banker_note:{note['note_id']}", "verification": "unverified", "asserted_by": "bank", "kind": "direct",
            "purposes": ["servicing", "relationship_management"]}
    return memory.add([{"role": "user", "content": note["text"]}], user_id=banker_id, infer=False, metadata=meta)["results"][0]["id"]


def write_playbook(memory, agent_id: str, playbook: dict) -> str:
    """Procedural memory: how the team resolves things, scoped to the agent, shared across customers."""
    meta = {"layer": "procedural", "audience": "internal", "source_system": "playbook", "source_ref": f"playbook:{playbook['id']}",
            "verification": "verified", "purposes": ["servicing", "relationship_management", "complaints"]}
    return memory.add([{"role": "user", "content": playbook["text"]}], agent_id=agent_id, infer=False, metadata=meta)["results"][0]["id"]

for banker_id, banker in vault_dataset["banker_memory"].items():
    for note in banker["notes"]:
        write_banker_note(vault, banker_id, note, directory)
for playbook in vault_dataset["playbooks"]:
    write_playbook(vault, "context_agent", playbook)

for r in vault_rows(vault, "banker_alex"):
    print(f"banker_alex   [{r['layer']:<10} mentions {r['mentions_customers'] or '-'}] {r['text'][:90]}")
for r in vault_rows(vault, "context_agent", scope="agent_id"):
    print(f"playbook      {r['text'][:100]}")

banker_alex   [preference mentions -] For customer briefs I want open commitments first, then what has changed since the last co
banker_alex   [episodic   mentions ['cust_priya']] I'm preparing Priya Sharma's annual review for Tuesday 6 October 2026. I want to raise the
playbook      When a merchant terminal fails a second time, book a technician visit rather than shipping another u
playbook      When a promised fee waiver is overdue, escalate to merchant operations, give the customer a new date
playbook      When a customer has an accountant on file, copy the accountant on loan and merchant paperwork.


Conversation-scoped scratch memory should not outlive the conversation. mem0 has a native expiry: a memory with an `expiration_date` in the past is hidden from `search()` and `get_all()` unless you ask for `show_expired=True`. It is not deleted, so a retention job still has to remove it (notebook 04).

In [36]:
from datetime import date, timedelta
yesterday = (date.today() - timedelta(days=1)).isoformat()
vault.add([{"role": "user", "content": "Currently comparing two quotes for the Newcastle terminal set-up."}],
          run_id="desk-session-8812", infer=False, expiration_date=yesterday, metadata={"layer": "episodic"})
print("visible:", len(vault.get_all(filters={"run_id": "desk-session-8812"}, top_k=10)["results"]),
      "| with show_expired:", len(vault.get_all(filters={"run_id": "desk-session-8812"}, top_k=10, show_expired=True)["results"]))

visible: 0 | with show_expired: 1


Finally, the whole build as one function. Notebooks 02 to 04 call it to build a fresh vault of their own.

In [37]:
def raw_sources(dataset: dict, vault_dataset: dict) -> list[dict]:
    """Every raw record the vault is built from: the seven conversations of notebook 03 plus the other sources."""
    sessions = [dict(s, kind="session", customer_id=cid) for cid, c in dataset["customers"].items() for s in c["sessions"]]
    return sessions + vault_dataset["sources"]


def build_vault(memory, dataset: dict, vault_dataset: dict, *, verbose: bool = False) -> dict:
    """Normalise, resolve and ingest every source in date order, after the systems of record. Returns the build log."""
    directory = vault_dataset["party_directory"]
    for customer_id, facts in vault_dataset["system_of_record"].items():
        ingest_system_of_record(memory, customer_id, facts)
    interactions = sorted((normalise(raw) for raw in raw_sources(dataset, vault_dataset)), key=lambda i: i.occurred_on)
    build = {"ingested": [], "quarantined": []}
    for interaction in interactions:
        customer_id, reason = resolve_customer(interaction, directory)
        if customer_id is None:
            build["quarantined"].append({"source_ref": interaction.source_ref, "reason": reason})
            continue
        log = ingest(memory, interaction, customer_id, directory["customers"][customer_id]) | {"customer_id": customer_id, "resolved_by": reason}
        build["ingested"].append(log)
        if verbose:
            print(f"{interaction.occurred_on}  {interaction.source_ref:<28} -> {customer_id:<12} wrote {len(log['written'])}, "
                  f"closed {len(log['closed'])}, superseded {len(log['superseded'])}, conflicts {len(log['conflicts'])}")
    for banker_id, banker in vault_dataset["banker_memory"].items():
        for note in banker["notes"]:
            write_banker_note(memory, banker_id, note, directory)
    for playbook in vault_dataset["playbooks"]:
        write_playbook(memory, "context_agent", playbook)
    return build

### What the vault adds to a memory

| | Part 4: mem0 as a memory | Part 5: the vault on mem0 |
|---|---|---|
| Sources | Conversations | Conversations, notes, cases, chats, events, emails, systems of record, through one adaptor |
| Whose memory | `user_id` given by the caller | Resolved from identifiers; third parties disambiguated; unresolvable items quarantined |
| What a memory is | A sentence | A typed record: layer, slot, value, who said it, direct or derived, verified or not, lineage, audience, purpose |
| Updates | Both facts kept, nothing marks the old one | Superseded with a dated link; reconfirmations counted; conflicts with a system of record kept and presented |
| Commitments | Sentences that mention a promise | Items with a due date and a state (open, overdue, closed) in any view |
| Time | Relative dates resolved against today | Event-time views as of any date |
| Scopes | user, agent, run | Customer, banker, agent playbooks, conversation with expiry |

## Part 6 · Recap

| The banker asks... | Which memory answers | Why |
|--------------------|----------------------|-----|
| "What did she just say?" | Working memory (`agent.messages`) | It is in the current run |
| "Where were we before the call dropped?" | Session memory (`SessionManager`) | Same conversation, resumed |
| "How does Priya want to be contacted?" | Long-term memory (mem0, `user_id`) | A fact from any past conversation |
| "How do we usually fix terminal faults?" | Long-term memory (mem0, `agent_id`) | A fact about the agent's job |

What you can now do:

- Tell the three kinds of memory apart, and pick the right tool for each.
- Configure mem0 open source on Bedrock with FAISS, with the two dimension keys matched and telemetry off.
- Run `add()`, `search()`, `get_all()`, `update()`, `delete()` and `history()` with correct 2.x syntax (`filters=`, not `user_id=`).
- Explain what the extractor does in its single LLM call, and why identical input is idempotent.
- Recognise the add-only behaviour of mem0 2.x and why contradictions are *your* problem to design for.
- Check that a metadata filter is actually implemented by your vector store before relying on it.
- Normalise many source systems into one interaction model with lineage, and resolve each interaction to one customer or quarantine it.
- Design a typed memory model (layers, slots, direct vs derived, verified vs unverified) and write it to mem0 with `infer=False`.
- Keep conflicting facts side by side with provenance, and compute the vault's state now and as of any past date.
- Separate customer, banker, agent and conversation memory, and expire scratch memory.

### Check your understanding

<details><summary>1. Why is Priya's September phone call stored with <code>observed_on = 2026-09-04</code> even though it was ingested today?</summary>

Because memory is about events. The date the call happened decides what was true when, which open items were overdue on which day, and how relative expressions like "on Wednesday" resolve. The date it was stored (record time) is kept separately, in mem0's own timestamps and history table, for audit.
</details>

<details><summary>2. An email arrives from an accountant who works for three of your customers and mentions none of their businesses. What should happen?</summary>

Quarantine it for a person to attach. The sender identifies a third party, not a customer, and attaching it to "the most likely" customer risks writing one customer's information into another's memory, which is a privacy breach that no later retrieval check can detect.
</details>

<details><summary>3. The CRM says 14 trucks and the customer says 16. Why does the vault not simply update the record to 16?</summary>

Because the customer's statement is unverified and the record is the bank's verified source. The MVP rule is to keep both with provenance, present them with context, and send the discrepancy for review; writing back to systems of record is out of scope and is a decision for their owners.
</details>

<details><summary>4. Why does the vault use an attribute vocabulary instead of asking a model "does this new memory replace an old one?"</summary>

Because naming the slot at extraction time makes an update a data comparison (same slot, different value) that is deterministic, cheap and auditable. The model-judged approach (notebook 04's reconciliation) still has a place for memories without a slot, but it costs a call per memory and can be wrong in both directions.
</details>

<details><summary>5. Your FAISS filter <code>{"layer": {"in": ["commitment"]}}</code> returns nothing. Does the customer have no commitments?</summary>

Not necessarily: FAISS does not implement operator filters and silently matches nothing. Use a plain list, <code>{"layer": ["commitment"]}</code>, or filter in application code, and test every filter against the store you run in production.
</details>

**Next: notebook 02.** So far we called mem0 by hand and built the vault's foundations. Now we wire memory into the Strands agent, compare three integration patterns, and then put the vault behind a policy-enforced memory service that many agents, on many channels, share safely.